# 🔎 EDA — Explorando a carteira

Com silver e gold prontos, dá pra fazer perguntas de verdade:

- Qual região concentra mais inadimplência?
- Qual assessoria recupera melhor?
- O método de pagamento diz algo sobre o devedor?
- Os pagamentos seguem alguma sazonalidade?

Vamos visualizar e ir comentando o que aparece.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

DATA   = Path.cwd().parents[0] / 'data'
SILVER = DATA / 'silver'
GOLD   = DATA / 'gold'

contracts = pd.read_parquet(SILVER / 'contracts.parquet')
payments  = pd.read_parquet(SILVER / 'payments.parquet')
features  = pd.read_parquet(GOLD   / 'contract_features.parquet')

contracts.shape, payments.shape, features.shape

((10000, 8), (100000, 11), (10000, 19))

## 1. Como está distribuído o status da cobrança?

In [2]:
fig = px.histogram(contracts, x='status_cobranca', color='regiao', barmode='group',
                   title='Status de cobrança por região')
fig.show()

## 2. Onde está o dinheiro inadimplente?

In [3]:
fig = px.sunburst(
    contracts, path=['regiao', 'status_cobranca'],
    values='valor_inadimplente',
    title='R$ inadimplente — região × status',
)
fig.show()

## 3. Performance por assessoria (apenas casos fechados)

In [4]:
closed = contracts[contracts['status_cobranca'].isin(['Acordo Firmado', 'Insucesso'])].copy()
closed['sucesso'] = (closed['status_cobranca'] == 'Acordo Firmado').astype(int)

perf = (
    closed.groupby('nome_assessoria')
          .agg(casos=('id_contrato', 'count'),
               taxa_sucesso=('sucesso', 'mean'),
               score_medio=('score_risco', 'mean'))
          .reset_index()
          .sort_values('taxa_sucesso', ascending=False)
)
perf

,nome_assessoria,casos,taxa_sucesso,score_medio
2,Nexus Mediação Financeira,988,0.704453,51.131579
0,Acerta Crédito Integrado,970,0.695876,51.794845
3,Vértice Asset & Cobrança,1490,0.685906,50.587919
1,Fênix Recuperação De Crédito,1508,0.672414,50.498674


In [5]:
fig = px.bar(perf, x='nome_assessoria', y='taxa_sucesso',
             color='score_medio', text='casos',
             title='Taxa de sucesso por assessoria')
fig.show()

## 4. Score de risco prevê insucesso?

In [6]:
fig = px.box(closed, x='status_cobranca', y='score_risco', color='status_cobranca',
             title='Score de risco × resultado da cobrança')
fig.update_layout(showlegend=False)
fig.show()

## 5. Forma de pagamento — Boleto, Pix, Débito Automático?

In [7]:
fig = px.histogram(payments, x='forma_pagamento', color='contemplado',
                   barmode='group', title='Forma de pagamento × contemplação')
fig.show()

## 6. Recebimentos ao longo do tempo

In [8]:
p = payments.dropna(subset=['data_pagamento']).copy()
p['mes'] = p['data_pagamento'].dt.to_period('M').astype(str)

ts = p.groupby(['mes', 'forma_pagamento'])['valor_pago'].sum().reset_index()
fig = px.area(ts, x='mes', y='valor_pago', color='forma_pagamento',
              title='Recebimentos mensais por forma de pagamento')
fig.show()

## 7. Correlações entre features numéricas

In [9]:
num = features.select_dtypes(include=np.number)
corr = num.corr()

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    colorscale='RdBu_r', zmin=-1, zmax=1,
))
fig.update_layout(title='Correlação entre features numéricas', height=600)
fig.show()

## 8. Assertivas de qualidade (gate do pipeline)

Estas asserções são parte do contrato: se qualquer uma falhar, o pipeline está quebrado.

In [10]:
report = []

def expect(name, ok, detail=''):
    report.append({'expectation': name, 'passed': ok, 'detail': detail})
    if not ok:
        raise AssertionError(f'{name} — {detail}')

expect('contracts.id_contrato único', contracts['id_contrato'].is_unique)
expect('contracts.status em domínio',
       set(contracts['status_cobranca']) <= {'Em Aberto', 'Acordo Firmado', 'Insucesso', 'Ajuizado'})
expect('contracts.score_risco em [0, 100]', contracts['score_risco'].between(0, 100).all())
expect('contracts.valor_inadimplente > 0', (contracts['valor_inadimplente'] > 0).all())
expect('contracts.regiao em domínio BR',
       set(contracts['regiao']) <= {'Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul'})
expect('payments.id_pagamento único', payments['id_pagamento'].is_unique)
expect('payments.valor_parcela > 0', (payments['valor_parcela'] > 0).all())
expect('payments → contracts (sem órfãos)',
       set(payments['id_contrato']) <= set(contracts['id_contrato']))
expect('contract_features 1:1 com contracts',
       features['id_contrato'].is_unique and len(features) == len(contracts))
expect('features.taxa_adimplencia em [0, 1]',
       features['taxa_adimplencia'].between(0, 1).all())

pd.DataFrame(report)

,expectation,passed,detail
0,contracts.id_contrato único,True,
1,contracts.status em domínio,True,
2,"contracts.score_risco em [0, 100]",True,
3,contracts.valor_inadimplente > 0,True,
4,contracts.regiao em domínio BR,True,
5,payments.id_pagamento único,True,
6,payments.valor_parcela > 0,True,
7,payments → contracts (sem órfãos),True,
8,contract_features 1:1 com contracts,True,
9,"features.taxa_adimplencia em [0, 1]",True,


In [11]:
REPORTS = Path.cwd().parents[1] / 'reports'
REPORTS.mkdir(exist_ok=True)
pd.DataFrame(report).to_csv(REPORTS / 'data_quality_report.csv', index=False)
print('Salvo em', REPORTS / 'data_quality_report.csv')

Salvo em /home/ivissonalves/Workspaces/previsia/previsia-api/reports/data_quality_report.csv
